In [2]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

## Folder Structure

In [9]:
data_path = Path.home() / 'projects/anomaly-detection/data/mvtec/'

data_structure = list(data_path.iterdir())
data_structure

[PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/capsule'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/toothbrush'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/cable'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/wood'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/screw'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/grid'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/bottle'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/transistor'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/license.txt'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/metal_nut'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/tile'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/leather'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/carpet'),
 PosixPath('/home/ali/projects/anomaly-detection/data/mvtec/pill'),
 PosixPath('

In [12]:
def build_mvtec_manifest(root: str | Path) -> pd.DataFrame:
    root = Path(root)

    rows = []

    for product_dir in root.iterdir():
        if not product_dir.is_dir():
            continue

        product = product_dir.name

        # train and test images
        for split in ["train", "test"]:
            split_dir = product_dir / split
            if not split_dir.exists():
                continue

            for defect_dir in split_dir.iterdir():
                if not defect_dir.is_dir():
                    continue

                defect_type = defect_dir.name

                for image_path in defect_dir.glob("*"):
                    if image_path.suffix.lower() not in [".png", ".jpg", ".jpeg", ".bmp"]:
                        continue

                    rows.append({
                        "path": image_path,
                        "product": product,
                        "split": split,
                        "defect_type": defect_type,
                        "kind": "image",
                        "is_good": defect_type == "good",
                        "is_defect": split == "test" and defect_type != "good",
                    })

        # ground truth masks
        gt_dir = product_dir / "ground_truth"
        if gt_dir.exists():
            for defect_dir in gt_dir.iterdir():
                if not defect_dir.is_dir():
                    continue

                defect_type = defect_dir.name

                for mask_path in defect_dir.glob("*"):
                    if mask_path.suffix.lower() not in [".png", ".jpg", ".jpeg", ".bmp"]:
                        continue

                    rows.append({
                        "path": mask_path,
                        "product": product,
                        "split": "ground_truth",
                        "defect_type": defect_type,
                        "kind": "mask",
                        "is_good": False,
                        "is_defect": False,
                    })

    df = pd.DataFrame(rows)

    if not df.empty:
        df["path"] = df["path"].astype(str)

    return df

In [14]:
df = build_mvtec_manifest(data_path)

In [15]:
df.head()

,path,product,split,defect_type,kind,is_good,is_defect
0,/home/ali/projects/anomaly-detection/data/mvte...,capsule,train,good,image,True,False
1,/home/ali/projects/anomaly-detection/data/mvte...,capsule,train,good,image,True,False
2,/home/ali/projects/anomaly-detection/data/mvte...,capsule,train,good,image,True,False
3,/home/ali/projects/anomaly-detection/data/mvte...,capsule,train,good,image,True,False
4,/home/ali/projects/anomaly-detection/data/mvte...,capsule,train,good,image,True,False


In [18]:
df.is_defect.value_counts()

is_defect
False    5354
True     1258
Name: count, dtype: int64

In [5]:

def load_images(folder):
    paths = list(Path(folder).glob("*.png"))
    imgs = [cv2.imread(str(p)) for p in paths]
    return np.array(imgs), np.array(paths)

In [ ]:

good_imgs, good_paths = load_images(data_path.joinpath("bottle/train/good"))

In [4]:
np.array(good_imgs).shape

(209, 900, 900, 3)

In [5]:
good_imgs[0].shape

(900, 900, 3)

In [8]:
print(f"Folder exists: {data_path.joinpath('bottle/train/good').exists()}")
print(f"Number of images found: {len(good_imgs)}")
print(f"Number of paths found: {len(good_paths)}")

# Check none failed to load (cv2.imread returns None silently on failure)
none_count = sum(1 for img in good_imgs if img is None)
print(f"Failed to load: {none_count}")

if good_imgs:
    print(f"First image shape: {good_imgs[0].shape}")

Folder exists: True
Number of images found: 209
Number of paths found: 209
Failed to load: 0


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()